In [0]:
T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_film = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
T_rfm = spark.table("cpx_dataanalytics_gold.customer360.c360_f_rfm_model")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")

In [0]:
import pyspark.sql.functions as F
T_trans = (
    T_trans
    .filter(F.col("TicketFLAG") == "Yes")

)

df_films = (
    T_trans
    .join(T_film, on = "film_id", how = "inner")
    .filter(F.col("language").isin('Hindi', 'Punjabi','Gujarati','Telugu', 'Tamil','Mandarin','Cantonese')) 
    .filter(F.col("theatrical_release_date") > "2024-01-01")
    .withColumn(
     "Offer Eligibility",
      F.when(
        F.col("language").isin('Hindi', 'Punjabi', 'Gujarati', 'Telugu', 'Tamil'),
        "Diwali"
         ).otherwise("CNY"))
)

# Attach RFM segment based on transaction date
df_films_rfm = (
    df_films
    .join(
        T_rfm.select(
            F.col("CDE_ID").alias("_rfm_cde_id"),
            F.col("SnapshotDateId").alias("_rfm_date"),
            F.col("Segment").alias("rfm_segment"),
        ),
        on=[
            df_films["CDE_ID"] == F.col("_rfm_cde_id"),
            df_films["VisitDateId"] == F.col("_rfm_date"),
        ],
        how="left",
    )
    .drop("_rfm_cde_id", "_rfm_date")
)

In [0]:
SEGMENTS = ["Champions", "High Value Lapsed", "Value Conscious Loyals", "At_Risk",
            "Potential Loyalist", "Promising", "Fleeting", "New"]

segment_aggs = []
for s in SEGMENTS:
    # Scene members who ARE CineClub
    scene_cc = (F.col("rfm_segment") == s) & (F.col("SCENEFLAG") == "Yes") & (F.col("CineClubFLAG") == "Yes")
    # Scene members who are NOT CineClub
    scene_only = (F.col("rfm_segment") == s) & (F.col("SCENEFLAG") == "Yes") & (F.col("CineClubFLAG") == "No")

    segment_aggs += [
        F.countDistinct(F.when(scene_cc,   F.col("CDE_ID"))).alias(f"customers_{s}_scene_cc"),
        F.sum(F.when(scene_cc,   F.col("Quantity")).otherwise(0)).alias(f"tickets_{s}_scene_cc"),
        F.countDistinct(F.when(scene_only, F.col("CDE_ID"))).alias(f"customers_{s}_scene_only"),
        F.sum(F.when(scene_only, F.col("Quantity")).otherwise(0)).alias(f"tickets_{s}_scene_only"),
    ]

def safe_div(numerator, denominator):
    """Return numerator/denominator, or null when denominator is 0 or null."""
    return F.when(F.coalesce(denominator, F.lit(0)) != 0, numerator / denominator)

# Step 4: Single aggregation — your originals plus segment counts
df_film_attendance = (
    df_films_rfm
    .groupBy("theatrical_release_date", "title_id", "title_name", "Offer Eligibility")
    .agg(
        F.sum("Quantity").alias("attendance"),
        F.sum("Net_Revenue").alias("Net_Revenue"),

        F.sum(F.when(F.col("SCENEFLAG") == "Yes", F.col("Quantity")).otherwise(0)).alias("attendance_scene"),
        F.sum(F.when(F.col("SCENEFLAG") == "Yes", F.col("Net_Revenue")).otherwise(0)).alias("Net_Revenue_scene"),

        F.sum(F.when(F.col("CineClubFLAG") == "Yes", F.col("Quantity")).otherwise(0)).alias("attendance_CC"),
        F.sum(F.when(F.col("CineClubFLAG") == "Yes", F.col("Net_Revenue")).otherwise(0)).alias("Net_Revenue_CC"),

        F.countDistinct(
            F.when((F.col("SCENEFLAG") == "Yes") & F.col("CDE_ID").isNotNull(), F.col("CDE_ID"))
        ).alias("Distinct_SCENE_Customers"),
        F.countDistinct("CDE_ID").alias("Distinct_Customers"),
        F.countDistinct(
            F.when(F.col("CineClubFLAG") == "Yes", F.col("CDE_ID"))
        ).alias("Distinct_CineClub_Customers"),

        # 👇 segment counts mixed right in
        *segment_aggs,
    )
    .withColumn("pct_CC_tickets_total",  safe_div(F.col("attendance_CC"),               F.col("attendance")))
    .withColumn("pct_CC_tickets_scene",  safe_div(F.col("attendance_CC"),               F.col("attendance_scene")))
    .withColumn("pct_CC_Members_scene",  safe_div(F.col("Distinct_CineClub_Customers"), F.col("Distinct_SCENE_Customers")))
    .filter(F.col("attendance") > 0)

)

In [0]:
df_film_attendance.display()

## Current State

In [0]:
cohort = (
    df_films_rfm
    .filter(F.col("CDE_ID").isNotNull())
    .select("CDE_ID")
    .distinct()
)

In [0]:
# Latest snapshot date in the RFM table
latest_rfm_date = T_rfm.agg(F.max("SnapshotDateId")).first()[0]

current_rfm = (
    T_rfm
    .filter(F.col("SnapshotDateId") == latest_rfm_date)
    .select(
        F.col("CDE_ID").alias("_cde"),
        F.col("Segment").alias("current_rfm_segment"),
    )
)

cohort_rfm = (
    cohort
    .join(current_rfm, cohort["CDE_ID"] == F.col("_cde"), "left")
    .drop("_cde")
)

In [0]:
# Latest subscriber snapshot
latest_sub_date = T_subscriber.agg(F.max("snapshot_date_id")).first()[0]

current_cc = (
    T_subscriber
    .filter(F.col("snapshot_date_id") == latest_sub_date)
    .filter(F.col("renewal_status").like("Active%"))      # active members today
    .select(
        F.col("cde_id").alias("_cde"),
        F.col("plan_id").alias("current_plan"),         # adjust to your plan column
        F.lit("Yes").alias("is_cc_today"),
    )
)

cohort_full = (
    cohort_rfm
    .join(current_cc, cohort_rfm["CDE_ID"] == F.col("_cde"), "left")
    .drop("_cde")
    .withColumn("is_cc_today", F.coalesce(F.col("is_cc_today"), F.lit("No")))
)

In [0]:
distribution = (
    cohort_full
    .groupBy("current_rfm_segment", "is_cc_today", "current_plan")
    .agg(F.countDistinct("CDE_ID").alias("customers"))
    .orderBy("current_rfm_segment", "is_cc_today", "current_plan")
)

In [0]:
distribution.display()

In [0]:
# df_film_attendance = (
#     df_films    
#     .groupBy("theatrical_release_date", "title_id", "title_name","Offer Eligibility")
#     .agg(
#         F.sum(F.col("Quantity")).alias("attendance"),
#         F.sum(F.col("Net_Revenue")).alias("Net_Revenue"),

#         F.sum(F.when(F.col("SCENEFLAG") == "Yes", F.col("Quantity")).otherwise(0)).alias("attendance_scene"),
#         F.sum(F.when(F.col("SCENEFLAG") == "Yes", F.col("Net_Revenue")).otherwise(0)).alias("Net_Revenue_scene"),

#         F.sum(F.when(F.col("CineClubFLAG") == "Yes", F.col("Quantity")).otherwise(0)).alias("attendance_CC"),
#         F.sum(F.when(F.col("CineClubFLAG") == "Yes", F.col("Net_Revenue")).otherwise(0)).alias("Net_Revenue_CC"),

#         F.countDistinct(
#             F.when(F.col("SCENEFLAG") == "Yes" & F.col("CDE_ID").isNotNull(), F.col("CDE_ID"))
#         ).alias("Distinct_SCENE_Customers"),

#         F.countDistinct("CDE_ID").alias("Distinct_Customers"),
#          F.countDistinct(
#             F.when(F.col("CineClubFLAG") == "Yes", F.col("CDE_ID"))
#         ).alias("Distinct_CineClub_Customers")
#     )
#     .withColumn("pct_CC_tickets_total", F.col("attendance_CC")/F.col("attendance"))
#     .withColumn("pct_CC_tickets_scene", F.col("attendance_CC")/F.col("attendance_scene"))
#     .withColumn("pct_CC_Members_scene", F.col("Distinct_CineClub_Customers")/F.col("Distinct_SCENE_Customers"))
        
#     .filter(F.col("attendance") > 0)
# )